In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [3]:
image = cv2.imread(r"C:\Users\PC-LAB1\img_processing_lab\images\image_with_shapes_2.jpeg")

if image is None:
    raise FileNotFoundError("Could not read the image. Check the path and file format.")

print(type(image))  # <class 'numpy.ndarray'>
print(image.shape)  # (height, width, channels)

<class 'numpy.ndarray'>
(577, 576, 3)


In [4]:
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
image_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
image_hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

print("Color:", image_rgb.shape)
print("Gray:", image_gray.shape)
print("HSV:", image_hsv.shape)

Color: (577, 576, 3)
Gray: (577, 576)
HSV: (577, 576, 3)


In [6]:

# ==========================================
# 1. HELPER FUNCTIONS
# ==========================================

def detect_and_measure_shapes(image_path):
    # Load image
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Could not load image at '{image_path}'")

    # Step 1: Preprocessing (Grayscale + Gaussian Blur + Thresholding)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # Otsu's thresholding automatically calculates optimal threshold value
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Step 2: Find contours (boundaries of shapes)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    output_img = img.copy()
    shape_data = []

    # Step 3: Iterate through each detected contour
    for idx, cnt in enumerate(contours, start=1):
        # Filter out tiny noise contours by minimum area
        contour_area = cv2.contourArea(cnt)
        if contour_area < 100:
            continue

        # Approximate shape geometry using Ramer-Douglas-Peucker algorithm
        perimeter = cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, 0.04 * perimeter, True)
        vertices = len(approx)

        shape_name = "Unknown"

        # --- SHAPE CLASSIFICATION LOGIC ---
        if vertices == 3:
            shape_name = "Triangle"
            
        elif vertices == 4:
            # Differentiate Square vs Rectangle using Aspect Ratio
            x, y, w, h = cv2.boundingRect(approx)
            aspect_ratio = float(w) / h
            # If aspect ratio is close to 1, it's a square; otherwise, rectangle
            if 0.95 <= aspect_ratio <= 1.05:
                shape_name = "Square"
            else:
                shape_name = "Rectangle"
                
        else:
            # Check circularity to distinguish circles from complex polygons
            # Circularity = 4 * pi * Area / (Perimeter^2) -> Ideally 1.0 for a perfect circle
            circularity = (4 * np.pi * contour_area) / (perimeter ** 2) if perimeter > 0 else 0
            if circularity > 0.75:
                shape_name = "Circle"
            else:
                shape_name = f"Polygon ({vertices}-sided)"

        # Save extracted shape information
        shape_data.append({
            "id": idx,
            "shape": shape_name,
            "area_px": round(contour_area, 2),
            "vertices": vertices
        })

        # --- VISUALIZATION ---
        # Draw green outline around detected contour
        cv2.drawContours(output_img, [cnt], -1, (0, 255, 0), 2)

        # Get centroid to place the label text centered on the shape
        M = cv2.moments(cnt)
        if M["m00"] != 0:
            cX = int(M["m10"] / M["m00"])
            cY = int(M["m01"] / M["m00"])
        else:
            cX, cY = approx[0][0]

        # Draw label on image
        label = f"{shape_name} ({int(contour_area)}px)"
        cv2.putText(output_img, label, (cX - 40, cY), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)

    return output_img, shape_data


# ==========================================
# 2. EXECUTION & RESULTS
# ==========================================
if __name__ == "__main__":
    input_path = r"C:\Users\PC-LAB1\img_processing_lab\images\image_with_shapes_2.jpeg"
    
    try:
        result_img, results = detect_and_measure_shapes(input_path)

        # Print detailed report table
        print(f"{'ID':<5} | {'Shape':<12} | {'Area (Pixels)':<15} | {'Vertices':<10}")
        print("-" * 50)
        for item in results:
            print(f"{item['id']:<5} | {item['shape']:<12} | {item['area_px']:<15} | {item['vertices']:<10}")

        # Save output image with labels and contours
        cv2.imwrite(r"C:\Users\PC-LAB1\img_processing_lab\output\labeled_shapes.jpg", result_img)
        print("\nProcessed image saved to output/labeled_shapes.jpg successfully!")

    except Exception as e:
        print("Error:", e)

ID    | Shape        | Area (Pixels)   | Vertices  
--------------------------------------------------
1     | Circle       | 3459.5          | 7         
2     | Rectangle    | 8554.0          | 4         
3     | Square       | 10201.0         | 4         
4     | Circle       | 10088.5         | 8         
5     | Triangle     | 1557.5          | 3         
6     | Square       | 10300.5         | 4         
7     | Rectangle    | 6147.0          | 4         
8     | Triangle     | 5525.5          | 3         
9     | Square       | 3419.0          | 4         
10    | Rectangle    | 2174.0          | 4         
11    | Circle       | 3463.5          | 7         
12    | Rectangle    | 8432.0          | 4         
13    | Square       | 2499.0          | 4         
14    | Triangle     | 5532.5          | 3         
15    | Circle       | 10091.0         | 7         
16    | Triangle     | 5485.0          | 3         

Processed image saved to output/labeled_shapes.jpg successfully!

In [8]:
image_1 = cv2.imread(r"C:\Users\PC-LAB1\img_processing_lab\images\image_morphology2.png")
image_2 = cv2.imread(r"C:\Users\PC-LAB1\img_processing_lab\images\morphology_images.png")
image_3 = cv2.imread(r"C:\Users\PC-LAB1\img_processing_lab\images\morphology3.png")

kernel = np.ones((5, 5), np.uint8)

eroded_image_1 = cv2.erode(image_1, kernel, iterations=1)
eroded_image_2 = cv2.erode(image_2, kernel, iterations=1)
eroded_image_3 = cv2.erode(image_3, kernel, iterations=1)

dilated_image_1 = cv2.dilate(image_1, kernel, iterations=1)
dilated_image_2 = cv2.dilate(image_2, kernel, iterations=1)
dilated_image_3 = cv2.dilate(image_3, kernel, iterations=1)

cv2.imwrite(r"C:\Users\PC-LAB1\img_processing_lab\output\image_morphology2_eroded.jpg", eroded_image_1)
cv2.imwrite(r"C:\Users\PC-LAB1\img_processing_lab\output\morphology_images_eroded.jpg", eroded_image_2)
cv2.imwrite(r"C:\Users\PC-LAB1\img_processing_lab\output\morphology3_eroded.jpg", eroded_image_3)

cv2.imwrite(r"C:\Users\PC-LAB1\img_processing_lab\output\image_morphology2_dilated.jpg", dilated_image_1)
cv2.imwrite(r"C:\Users\PC-LAB1\img_processing_lab\output\morphology_images_dilated.jpg", dilated_image_2)
cv2.imwrite(r"C:\Users\PC-LAB1\img_processing_lab\output\morphology3_dilated.jpg", dilated_image_3)

print("Images saved successfully!")

Images saved successfully!
